# Project 21 — Agent Runtime (plan → tool → execute → observe, with retries/timeouts/permissions)

```
10 scripted tasks (calc / fs / python / db / http / multistep / memory-compare / flaky / slow / forbidden)
  -> rule-based planner (documented stand-in for the LLM) -> permission gate -> tool executor
  -> observe -> scratchpad memory -> retry-with-backoff / per-call timeout / max-steps guard
  -> 5 runtime configs: full vs no-retry vs no-memory vs tight-timeout vs strict-perms
  -> metrics: success rate, steps, retries, timeouts, denials; artifacts: tasks.csv + agent.csv + agent.png
```

Goal: learn what the loop around the model must enforce — tools fail, hang, and overreach, so the
runtime owns retries, timeouts, state, memory, and permissions. Ablations remove one guard at a time.

Learning objectives:
1. Draw LLM → tool → exec → observe → memory and name what each guard (retry/timeout/perm) catches.
2. State why retries rescue flaky tools, timeouts bound slow ones, and permissions gate side effects.
3. Show a task that fails without memory (compare needs two distant observations, not one).
4. Read the config table and argue which guard each workload pays for.
  Key lesson: the model proposes, the runtime disposes — reliability lives in the loop, not the prompt.


In [ ]:
import os # need os for sandbox paths + traversal guard
import re # need re for parsing tool outputs + calculator token guard
import ast # need ast for safe expression parsing (calculator + import scan)
import math # need math for the calculator function allowlist
import time # need time for wall clocks + backoff + slow-tool simulation
import csv # need csv to write tasks.csv + agent.csv
import json # need json to parse the mock http bodies
import concurrent.futures as cf # need threads to enforce per-call timeouts
import numpy as np # need numpy for means of steps/retries
import matplotlib # need matplotlib for figures
matplotlib.use('Agg') # force headless PNG backend (no display needed)
import matplotlib.pyplot as plt # need pyplot for success/steps/guards figure
SEED = 21 # fixed seed: deterministic task order + flaky counters
import random # need random for seeded task shuffle (stdlib only beyond this line)
random.seed(SEED) # seed stdlib
np.random.seed(SEED) # seed numpy
SB = os.path.realpath('./results/sandbox') # sandbox root, resolved (traversal guard compares against this)
os.makedirs('./results', exist_ok=True) # create results/ for CSVs + PNG
os.makedirs(SB, exist_ok=True) # create the tool sandbox dir
print(f"sandbox={SB} seed={SEED}") # env line for the README


In [ ]:
# ---- tools: each returns (ok: bool, payload: str, latency_s: float); pure + deterministic ----
import time as _t # local alias (keeps cell-1 import list canonical)
def t_calc(expr): # safe calculator: AST allowlist, no names except pi/e + math funcs
    try: # parse first (syntax errors -> tool error, never a crash)
        tree = ast.parse(expr, mode='eval') # expression AST
    except SyntaxError as e: # bad syntax
        return False, f"syntax: {e}", 0.0 # tool-level error
    allowed = (ast.Expression, ast.BinOp, ast.UnaryOp, ast.Call, ast.Constant, ast.Name, ast.Load, # structural nodes
               ast.Add, ast.Sub, ast.Mult, ast.Div, ast.FloorDiv, ast.Mod, ast.Pow, ast.UAdd, ast.USub) # operators
    funcs = {'sqrt': math.sqrt, 'log': math.log, 'sin': math.sin, 'cos': math.cos, 'exp': math.exp} # callable allowlist
    for nd in ast.walk(tree): # audit every node
        if not isinstance(nd, allowed): # anything else (Attribute, Subscript, Lambda, Import...)
            return False, f"blocked node: {type(nd).__name__}", 0.0 # reject (injection-safe)
        if isinstance(nd, ast.Name) and nd.id not in ('pi', 'e') and nd.id not in funcs: # unknown name
            return False, f"blocked name: {nd.id}", 0.0 # reject
        if isinstance(nd, ast.Call) and not (isinstance(nd.func, ast.Name) and nd.func.id in funcs): # indirect call
            return False, "blocked call", 0.0 # reject
    try: # evaluate in a bare namespace (no builtins at all)
        val = eval(compile(tree, '<calc>', 'eval'), {'__builtins__': {}}, {'pi': math.pi, 'e': math.e, **funcs}) # safe eval
    except ZeroDivisionError: # divide by zero
        return False, "div by zero", 0.0 # tool-level error
    except Exception as e: # overflow etc.
        return False, f"eval: {e}", 0.0 # tool-level error
    return True, repr(float(val)), 0.0 # numeric result as string
def t_fs_write(name, text): # sandboxed write: name must stay inside SB
    p = os.path.realpath(os.path.join(SB, name)) # resolve symlinks + '..'
    if not p.startswith(SB + os.sep): # escape attempt
        return False, "denied: path escapes sandbox", 0.0 # block traversal
    with open(p, 'w') as f: # write inside sandbox
        f.write(text) # persist text
    return True, f"wrote {len(text)} chars", 0.0 # ack with size
def t_fs_read(name): # sandboxed read (same gate)
    p = os.path.realpath(os.path.join(SB, name)) # resolve
    if not p.startswith(SB + os.sep): # escape attempt
        return False, "denied: path escapes sandbox", 0.0 # block traversal
    if not os.path.exists(p): # missing file
        return False, "not found", 0.0 # tool-level error
    with open(p) as f: # read inside sandbox
        return True, f.read(), 0.0 # file contents
def t_python(code): # restricted exec: arithmetic/data ops only, result via `out = ...`
    if 'import' in code or 'open(' in code or '__' in code: # textual pre-scan (defense in depth)
        return False, "blocked: import/open/dunder", 0.0 # reject before exec
    box = {'__builtins__': {'abs': abs, 'min': min, 'max': max, 'sum': sum, 'range': range, 'len': len}} # tiny builtins
    try: # exec with no imports available
        exec(compile(code, '<py>', 'exec'), box) # run snippet
    except Exception as e: # any runtime failure
        return False, f"exec: {e}", 0.0 # tool-level error
    if 'out' not in box: # contract: snippet must set out
        return False, "no `out` set", 0.0 # contract error
    return True, repr(box['out']), 0.0 # return the output value
DB = {} # tiny dict-backed database (table = key prefix before ':')
def t_kv_put(k, v): # store a float under key
    try: # validate numeric
        DB[k] = float(v) # coerce to float
    except ValueError: # non-numeric
        return False, "kv: value must be numeric", 0.0 # tool-level error
    return True, f"ok:{DB[k]}", 0.0 # ack echoes the stored value
def t_kv_get(k): # fetch one key
    if k not in DB: # missing
        return False, "kv: missing key", 0.0 # tool-level error
    return True, repr(DB[k]), 0.0 # the value
def t_kv_sum(prefix): # sum all keys starting with prefix (aggregate query)
    vs = [v for k, v in DB.items() if k.startswith(prefix)] # scan matching keys
    if not vs: # empty aggregate
        return False, "kv: no keys with prefix", 0.0 # tool-level error
    return True, repr(float(sum(vs))), 0.0 # the total
HTTP = {'weather://halden': '{"temp_c": 21}', 'fx://usd-eur': '{"rate": 0.92}'} # offline mock web (no network)
def t_http_get(url): # mocked GET: known URLs 200, everything else 404
    if url not in HTTP: # unknown
        return False, "http 404: unknown url", 0.05 # small latency even on miss
    return True, HTTP[url], 0.05 # body + simulated latency
def t_slow_echo(delay, msg): # simulates a hanging tool (timeout probe)
    _t.sleep(float(delay)) # block the worker thread
    return True, f"echo:{msg}", float(delay) # late answer
FLAKY = {'n': 0} # call counter for the flaky tool (reset per task)
def t_flaky_add(a, b, fail_first): # fails fail_first times, then adds (retry probe)
    FLAKY['n'] += 1 # count this attempt
    if FLAKY['n'] <= int(fail_first): # still in the failure window
        return False, f"flaky: attempt {FLAKY['n']} failed", 0.0 # transient error
    return True, repr(float(a) + float(b)), 0.0 # eventual success
TOOLS = {'calculator': t_calc, 'fs_write': t_fs_write, 'fs_read': t_fs_read, 'python': t_python, # tool registry
         'kv_put': t_kv_put, 'kv_get': t_kv_get, 'kv_sum': t_kv_sum, 'http_get': t_http_get, # registry cont.
         'slow_echo': t_slow_echo, 'flaky_add': t_flaky_add} # registry cont.
print(f"tools={sorted(TOOLS)}") # registry card


In [ ]:
# ---- tool unit tests: contracts, sandboxing, error paths (must all pass before the loop runs) ----
ok, r, _ = t_calc('(17*23+41)') # precedence probe: 17*23+41 = 432
assert ok and abs(float(r) - 432.0) < 1e-9 # arithmetic contract
ok, r, _ = t_calc('1/0') # divide-by-zero probe
assert not ok and 'zero' in r # error, not exception
ok, r, _ = t_calc('__import__("os").system("x")') # injection probe
assert not ok and 'blocked' in r # AST gate rejects it
ok, r, _ = t_calc('sqrt(16)+sin(0)') # allowlist funcs probe
assert ok and abs(float(r) - 4.0) < 1e-9 # sqrt(16)=4, sin(0)=0
ok, r, _ = t_fs_write('t.txt', 'hello 42') # roundtrip write
assert ok # write acks
ok, r, _ = t_fs_read('t.txt') # roundtrip read
assert ok and r == 'hello 42' # contents match
ok, r, _ = t_fs_write('../evil.txt', 'x') # traversal probe
assert not ok and 'denied' in r # sandbox gate blocks it
ok, r, _ = t_fs_read('/etc/hostname') # absolute-path probe
assert not ok and ('denied' in r or 'not found' in r) # never leaves the sandbox
ok, r, _ = t_python('out = sum([i*i for i in range(10)])') # snippet probe: 0+1+...+81 = 285
assert ok and r == '285' # exec contract
ok, r, _ = t_python('import os\noute = 1') # import probe (escaped newline keeps one-line-per-comment style)
assert not ok and 'blocked' in r # no imports in the box
ok, r, _ = t_python('out = open("x").read()') # open probe
assert not ok and 'blocked' in r # no file access in the box
DB.clear() # fresh database for kv tests
ok, _, _ = t_kv_put('m:a', '7') # put probe
assert ok # stored
ok, r, _ = t_kv_get('m:a') # get probe
assert ok and r == '7.0' # fetched
ok, _, _ = t_kv_put('m:b', '5') # second key
assert ok # stored
ok, r, _ = t_kv_sum('m:') # aggregate probe
assert ok and r == '12.0' # 7+5 = 12
ok, r, _ = t_kv_get('m:zz') # missing-key probe
assert not ok # error, not KeyError
ok, r, _ = t_http_get('weather://halden') # mock 200 probe
assert ok and json.loads(r)['temp_c'] == 21 # body parses
ok, r, _ = t_http_get('http://nope') # mock 404 probe
assert not ok and '404' in r # offline-safe miss
FLAKY['n'] = 0 # reset counter
ok, _, _ = t_flaky_add(1, 2, 2) # fail-window probe 1
assert not ok # first call fails
ok, _, _ = t_flaky_add(1, 2, 2) # fail-window probe 2
assert not ok # second call fails
ok, r, _ = t_flaky_add(1, 2, 2) # third call succeeds
assert ok and r == '3.0' # 1+2 = 3 after the window
print('tool unit-tests: all pass') # gate card (loop starts below)


In [ ]:
# ---- runtime: permission gate + timeout executor + retry loop + scratchpad memory ----
EXEC = cf.ThreadPoolExecutor(max_workers=4) # shared worker pool for tool calls (timeout enforcement)
def call_tool(name, args, timeout): # one guarded tool call: perm gate -> exec with timeout
    if name not in TOOLS: # unknown tool (planner hallucination)
        return False, f"unknown tool: {name}", 0.0, 'error' # fail closed, classified
    if name in PERMS and not PERMS[name]: # permission gate (set per config below)
        return False, f"denied by policy: {name}", 0.0, 'denied' # blocked before exec
    fut = EXEC.submit(TOOLS[name], *args) # run on a worker thread
    t0 = time.time() # wall clock for this call
    try: # bounded wait
        ok, pay, _ = fut.result(timeout=timeout) # enforce per-call timeout
    except cf.TimeoutError: # too slow
        return False, f"timeout after {timeout}s: {name}", time.time() - t0, 'timeout' # classified timeout
    return ok, pay, time.time() - t0, ('ok' if ok else 'error') # classified outcome
def run_task(task, cfg): # plan -> act -> observe loop with retries + memory window
    global PERMS # permission map is config-scoped
    PERMS = cfg['perms'] # install this config's policy
    FLAKY['n'] = 0 # reset the flaky counter per task (deterministic)
    DB.clear() # fresh database per task (isolation)
    mem = [] # scratchpad: list of (thought, action, obs) dicts
    retries = timeouts = denials = errors = steps = 0 # counters
    t0 = time.time() # task wall clock
    answer, failed = None, False # terminal state
    for _ in range(cfg['max_steps']): # bounded loop (no infinite agents)
        vis = mem if cfg['memory'] else mem[-1:] # memory window: full history vs last obs only
        thought, name, args = task['planner'](task, vis, len(mem)) # planner sees window; stage = full count
        if name == 'finish': # planner declares done
            answer = args # the proposed answer
            break # exit loop
        ok, pay, dt, kind = call_tool(name, args, cfg['timeout']) # guarded execution
        steps += 1 # count the attempt
        attempt = 0 # retry counter for this action
        while not ok and attempt < cfg['retries']: # retry-with-backoff on any failure class
            attempt += 1 # count it
            retries += 1 # meter it
            time.sleep(0.005 * attempt) # tiny backoff (keeps runtime fast, still visible)
            ok, pay, dt2, kind = call_tool(name, args, cfg['timeout']) # same action again
            dt += dt2 # accumulate latency
        if kind == 'timeout': # classify for the ledger
            timeouts += 1 # meter timeouts
        if kind == 'denied': # classify for the ledger
            denials += 1 # meter denials
        if not ok: # exhausted (or zero) retries
            errors += 1 # meter terminal errors
            failed = True # task failed
            answer = f"FAILED: {pay}" # record cause
            break # stop the loop
        mem.append({'thought': thought, 'action': f"{name}{tuple(args)}", 'obs': pay}) # observe: append to scratchpad
    else: # max_steps exhausted without finish
        failed = True # over budget
        answer = "FAILED: max_steps" # record cause
    good = (not failed) and task['check'](task, answer, mem) # verify answer against task checker
    return {'task': task['name'], 'success': int(good), 'steps': steps, 'retries': retries, # result record
            'timeouts': timeouts, 'denials': denials, 'errors': errors, # guard counters
            'wall_ms': round((time.time() - t0) * 1000, 1), 'answer': str(answer)[:60]} # latency + short answer
PERMS = {} # placeholder (installed per config inside run_task)
print('runtime ready: gate + timeout + retry + memory') # runtime card


In [ ]:
# ---- 10 tasks (planner = rule-based LLM stand-in) x 5 configs; save CSVs + figure ----
def last_obs(mem, i=1): # helper: obs i steps back ('' when missing, e.g. under no-memory)
    return mem[-i]['obs'] if len(mem) >= i else '' # safe accessor for planners
def P_calc(t, m, s): # planner: one calculator call then finish
    return ('compute directly', 'calculator', ['(17*23+41)']) if s == 0 else ('done', 'finish', last_obs(m)) # stage machine on s
def P_fs(t, m, s): # planner: write then read then finish
    if s == 0: # stage 0
        return ('persist the note', 'fs_write', ['note.txt', 'hello agent 42']) # write
    if s == 1: # stage 1
        return ('read it back', 'fs_read', ['note.txt']) # read
    return ('verify contents', 'finish', last_obs(m)) # stage 2: answer = file contents
def P_db(t, m, s): # planner: three puts then a prefix sum
    ks = ['r:x', 'r:y', 'r:z'] # key names
    vs = ['4', '8', '15'] # values: total 27
    if s < 3: # stages 0..2
        return (f"store {ks[s]}", 'kv_put', [ks[s], vs[s]]) # put next
    if s == 3: # stage 3
        return ('aggregate', 'kv_sum', ['r:']) # sum prefix
    return ('done', 'finish', last_obs(m)) # answer = total
def P_py(t, m, s): # planner: one snippet then finish
    return ('squares via snippet', 'python', ['out = sum([i*i for i in range(10)])']) if s == 0 else ('done', 'finish', last_obs(m)) # 285
def P_multi(t, m, s): # planner: http temp -> C-to-F -> write -> read back (chain needs only the last obs each step)
    if s == 0: # stage 0: fetch weather
        return ('get temp', 'http_get', ['weather://halden']) # -> {"temp_c": 21}
    if s == 1: # stage 1: convert using last obs
        c = json.loads(last_obs(m))['temp_c'] # parse 21 from the previous observation
        return ('c to f', 'calculator', [f"({c}*9/5+32)"]) # -> 69.8
    if s == 2: # stage 2: persist using last obs
        return ('save forecast', 'fs_write', ['f.txt', f"{last_obs(m)}F"]) # write '69.8F'
    if s == 3: # stage 3: read back the file
        return ('read forecast', 'fs_read', ['f.txt']) # -> '69.8F'
    return ('verify file', 'finish', last_obs(m)) # answer = file contents
def P_cmp(t, m, s): # planner: store alpha + beta, then add BOTH (stage 2 needs alpha, 2 obs back: distant memory)
    if s == 0: # stage 0
        return ('store alpha', 'kv_put', ['c:alpha', '7']) # obs -> 'ok:7.0'
    if s == 1: # stage 1
        return ('store beta', 'kv_put', ['c:beta', '5']) # obs -> 'ok:5.0'
    if s == 2: # stage 2: recall both values from the visible window
        a = last_obs(m, 2).split(':')[-1] # alpha, 2 back ('' when the window hides it)
        b = last_obs(m, 1).split(':')[-1] # beta, 1 back
        try: # both present?
            expr = f"({float(a)}+{float(b)})" # e.g. (7.0+5.0)
        except ValueError: # alpha forgotten (no-memory window)
            expr = "(5.0+5.0)" # blind guess: reuses beta twice
        return ('add recalled values', 'calculator', [expr]) # compute the guess
    return ('done', 'finish', last_obs(m)) # stage 3: answer = sum
def P_flaky(t, m, s): # planner: flaky add then finish (runtime retries carry it)
    return ('add with retry cover', 'flaky_add', [6, 7, 2]) if s == 0 else ('done', 'finish', last_obs(m)) # 13.0
def P_slow(t, m, s): # planner: one slow call then finish
    return ('slow fetch', 'slow_echo', [0.4, 'late-data']) if s == 0 else ('done', 'finish', last_obs(m)) # 0.4s job
def P_perm(t, m, s): # planner: write a config file (forbidden under strict policy)
    return ('save config', 'fs_write', ['cfg.txt', 'mode=prod']) if s == 0 else ('done', 'finish', last_obs(m)) # policy decides
def P_bad(t, m, s): # planner: hallucinates a nonexistent tool (graceful-degradation probe)
    return ('teleport data', 'teleport', ['x']) if s == 0 else ('done', 'finish', last_obs(m)) # must fail closed
TASKS = [ # (name, planner, checker): checkers close the loop on ground truth
    {'name': 'calc', 'planner': P_calc, 'check': lambda t, a, m: abs(float(a) - 432.0) < 1e-9}, # 17*23+41
    {'name': 'fs', 'planner': P_fs, 'check': lambda t, a, m: a == 'hello agent 42'}, # roundtrip exact
    {'name': 'db', 'planner': P_db, 'check': lambda t, a, m: abs(float(a) - 27.0) < 1e-9}, # 4+8+15
    {'name': 'python', 'planner': P_py, 'check': lambda t, a, m: a == '285'}, # sum of squares
    {'name': 'multistep', 'planner': P_multi, 'check': lambda t, a, m: a == '69.8F'}, # http -> calc -> write -> read
    {'name': 'compare', 'planner': P_cmp, 'check': lambda t, a, m: abs(float(a) - 12.0) < 1e-9}, # 7+5 needs memory
    {'name': 'flaky', 'planner': P_flaky, 'check': lambda t, a, m: abs(float(a) - 13.0) < 1e-9}, # 6+7 after 2 fails
    {'name': 'slow', 'planner': P_slow, 'check': lambda t, a, m: a == 'echo:late-data'}, # survives iff timeout > 0.4
    {'name': 'perm', 'planner': P_perm, 'check': lambda t, a, m: 'wrote' in a}, # survives iff fs_write allowed
    {'name': 'badtool', 'planner': P_bad, 'check': lambda t, a, m: False}] # always fails: planner error must stay contained
OPEN = {n: True for n in TOOLS} # open policy: every tool allowed
STRICT = {n: True for n in TOOLS} # strict starts open...
STRICT.update({'fs_write': False}) # ...then denies persistence (compute stays allowed: blast radius = writers)
CFGS = [ # 5 runtime configs: full vs one guard removed each
    {'name': 'full', 'retries': 3, 'timeout': 2.0, 'memory': True, 'perms': OPEN, 'max_steps': 12}, # all guards on
    {'name': 'no-retry', 'retries': 0, 'timeout': 2.0, 'memory': True, 'perms': OPEN, 'max_steps': 12}, # retries off
    {'name': 'no-memory', 'retries': 3, 'timeout': 2.0, 'memory': False, 'perms': OPEN, 'max_steps': 12}, # scratchpad off
    {'name': 'tight-timeout', 'retries': 3, 'timeout': 0.1, 'memory': True, 'perms': OPEN, 'max_steps': 12}, # 0.1s budget
    {'name': 'strict', 'retries': 3, 'timeout': 2.0, 'memory': True, 'perms': STRICT, 'max_steps': 12}] # deny writes
all_rows, summary = [], [] # per-task rows + per-config summary rows
for cfg in CFGS: # per runtime config
    got = [] # task results under this config
    for tk in TASKS: # per task (deterministic order)
        r = run_task(tk, cfg) # execute the loop
        r['config'] = cfg['name'] # tag config
        all_rows.append(r) # store
        got.append(r) # per-config list
        print(f"{cfg['name']:>13} {r['task']:<9} ok={r['success']} steps={r['steps']} retries={r['retries']} to={r['timeouts']} deny={r['denials']} {r['wall_ms']}ms") # ledger line
    sr = float(np.mean([g['success'] for g in got])) # success rate
    summary.append({'config': cfg['name'], 'success_rate': round(sr, 4), # summary row
                    'avg_steps': round(float(np.mean([g['steps'] for g in got])), 2), # mean tool calls
                    'total_retries': sum(g['retries'] for g in got), # retry spend
                    'total_timeouts': sum(g['timeouts'] for g in got), # timeout count
                    'total_denials': sum(g['denials'] for g in got)}) # denial count
S = {s['config']: s for s in summary} # config -> summary map
assert S['full']['success_rate'] == 0.9 # full passes 9/10 (only hallucinated tool fails, by design)
assert S['no-retry']['success_rate'] == 0.8 # flaky dies without retries (+ badtool)
assert S['no-memory']['success_rate'] == 0.8 # compare dies without distant memory (+ badtool)
assert S['tight-timeout']['success_rate'] == 0.8 # slow job exceeds 0.1s (+ badtool)
assert S['strict']['success_rate'] == 0.6 # denying writes kills fs AND multistep AND perm (+ badtool): blast radius = writers
assert S['full']['total_retries'] >= 2 # flaky alone spends 2 retries under full
assert S['tight-timeout']['total_timeouts'] >= 1 # slow job must trip the timeout guard
assert S['strict']['total_denials'] >= 1 # perm task must trip the permission gate
assert S['full']['success_rate'] > S['no-retry']['success_rate'] # retries buy reliability (measured)
with open('./results/tasks.csv', 'w', newline='') as f: # save per-task ledger
    w = csv.DictWriter(f, fieldnames=['config', 'task', 'success', 'steps', 'retries', 'timeouts', 'denials', 'errors', 'wall_ms', 'answer']) # header
    w.writeheader() # header once
    w.writerows(all_rows) # 50 rows (5 configs x 10 tasks)
print('saved ./results/tasks.csv') # confirm artifact
with open('./results/agent.csv', 'w', newline='') as f: # save per-config summary
    w = csv.DictWriter(f, fieldnames=list(summary[0].keys())) # header from keys
    w.writeheader() # header once
    w.writerows(summary) # 5 rows
print('saved ./results/agent.csv') # confirm artifact
EXEC.shutdown(wait=True) # stop the worker pool (clean exit, no dangling threads)
fig, ax = plt.subplots(1, 3, figsize=(15, 4)) # three panels: success, cost, guards
ax[0].bar([s['config'] for s in summary], [s['success_rate'] for s in summary]) # success bars
ax[0].set_ylim(0, 1.05) # rate range
ax[0].set_ylabel('success rate') # y label
ax[0].set_title('P21: each missing guard kills its task class') # panel title
ax[0].tick_params(axis='x', rotation=15) # tilt labels
ax[1].bar([s['config'] for s in summary], [s['avg_steps'] for s in summary], label='avg steps') # steps bars
ax[1].bar([s['config'] for s in summary], [s['total_retries'] for s in summary], alpha=0.4, label='retries') # retry overlay
ax[1].set_title('P21: tool-call cost per config') # panel title
ax[1].legend() # show series
ax[1].tick_params(axis='x', rotation=15) # tilt labels
ax[2].bar([s['config'] for s in summary], [s['total_timeouts'] for s in summary], label='timeouts') # timeout bars
ax[2].bar([s['config'] for s in summary], [s['total_denials'] for s in summary], bottom=[s['total_timeouts'] for s in summary], label='denials') # stacked denials
ax[2].set_title('P21: guard trips per config') # panel title
ax[2].legend() # show series
ax[2].tick_params(axis='x', rotation=15) # tilt labels
fig.tight_layout() # avoid overlap
fig.savefig('./results/agent.png', dpi=100) # save figure
print('saved ./results/agent.png') # confirm artifact


Cleanup / next steps:
- Artifacts: `results/tasks.csv` (50-row config × task ledger), `results/agent.csv` (per-config summary),
  `results/agent.png`, plus `results/sandbox/` (the agent's file side effects, safe to delete).
- Rerun anywhere: stdlib + numpy only, no network (mock http), sleeps ≤ 0.4 s, deterministic seeds.
- Next: Session 8 (P22 distributed training → P23 production platform → P24 sports-AI capstone) puts
  P19's vision, P20's retrieval, and this loop behind one API: detect → track → retrieve → analyze.
